## Para esta tarea voy a cargar el dataset para CIFAR y voy a correr 3 arquitecturas diferentes de redes neuronales

**imports**

In [ ]:
import random

import torch.nn as nn
import pandas as pd
import torch
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

**CUDA**

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

True

**Cargamos dataset y pasamos a dataloaders**

In [10]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

batch_size = 32

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # A parte de pasar a tensores normalizamos, no nos pide la actividad pero quiero hacerlo
])

train_data = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [4]:
print(len(train_data))
print(len(test_data))

50000
10000


In [12]:
# Definimos nuestras variables que usaremos para las funciones de entrenar

IMG_CHANNELS = 3
IMG_HEIGHT = 32
IMG_WIDTH = 32

**Funcion de entrenamiento**


In [13]:
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

In [18]:
def train(_model, _train_loader, _test_loader, _criterion, _optimizer, _num_epochs):
    res = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': [],
    }

    iterator = tqdm(range(_num_epochs), desc="Training", unit="epoch")

    for _ in iterator:
        # ===== TRAIN =====
        _model.train()
        train_loss = 0.0
        train_acc = 0.0

        for X_batch, y_batch in _train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            _optimizer.zero_grad()
            outputs = _model(X_batch)
            loss = _criterion(outputs, y_batch)
            loss.backward()
            _optimizer.step()

            train_loss += loss.item() * X_batch.size(0)
            train_acc += get_batch_accuracy(outputs, y_batch, len(_train_loader.dataset))

        epoch_train_loss = train_loss / len(_train_loader.dataset)

        # ===== TEST =====
        _model.eval()
        test_loss = 0.0
        test_acc = 0.0

        with torch.no_grad():
            for X_test, y_test in _test_loader:
                X_test = X_test.to(device)
                y_test = y_test.to(device)

                test_outputs = _model(X_test)
                loss = _criterion(test_outputs, y_test)

                test_loss += loss.item() * X_test.size(0)
                test_acc += get_batch_accuracy(test_outputs, y_test, len(_test_loader.dataset))

        epoch_test_loss = test_loss / len(_test_loader.dataset)

        iterator.set_postfix(
            train_loss=f"{epoch_train_loss:.4f}",
            train_acc=f"{train_acc:.4f}",
            test_loss=f"{epoch_test_loss:.4f}",
            test_acc=f"{test_acc:.4f}",
        )

        res['train_loss'].append(epoch_train_loss)
        res['train_acc'].append(train_acc)
        res['test_loss'].append(epoch_test_loss)
        res['test_acc'].append(test_acc)

    return res

**Funcion de evalaucion de la red neuronal usando test, NO es necesaria por la indicacion de la tarea**

**Funcion para imprimir los plots (Loss curves y Acc curves)**

In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curves(train_history):
    epochs = range(1, len(train_history['train_loss']) + 1)

    # Loss
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_history['train_loss'], label='Train Loss')
    plt.plot(epochs, train_history['test_loss'], label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Curvas de pérdida')
    plt.legend()
    plt.show()

    # Accuracy
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_history['train_acc'], label='Train Accuracy')
    plt.plot(epochs, train_history['test_acc'], label='Test Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Curvas de accuracy')
    plt.legend()
    plt.show()

**Definimos 3 arquitecturas para las CNNs**

Ya pregunte al profe y podemos usar nn.Sequential en vez de class convolution y referenciarlo 

**Primer arquitectura de CNN**

In [ ]:
model_cnn = nn.Sequential(
    nn.Conv2d(IMG_CHANNELS, 25, 3, stride=1, padding=1),  # 25 x 28 x 28
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 25 x 14 x 14

    nn.Conv2d(25, 50, 3, stride=1, padding=1),  # 50 x 14 x 14
    nn.ReLU(),
    nn.Dropout(.2),
    nn.MaxPool2d(2, stride=2),  # 50 x 7 x 7

    nn.Conv2d(50, 75, 3, stride=1, padding=1),  # 75 x 7 x 7
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),  # 75 x 3 x 3

    nn.Flatten(),
    nn.Linear(75 * 3 * 3, 512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512, 24)
)

model_cnn = model_cnn.to(device)

In [ ]:
epochs = 20
loss_function = nn.CrossEntropyLoss() # Esta es nuestra funcion de perdida apropiada al problema ya que es para clasificacion multiclase
optimizer = Adam(model_cnn.parameters()) # Utilizaremos el optimizador Adam

cnn_res = train(model_cnn, train_loader, loss_function, optimizer, epochs)

In [ ]:
cnn_test_loss, cnn_test_acc = test(model_cnn, test_loader, loss_function)

In [ ]:
print(cnn_test_loss, cnn_test_acc)

**Segunda arquitectura**

Esta la vamos a definir con el class convolution para probar otra manera de implementar la CNN aunque es la que menos me gusta

In [ ]:
class ConvolutionalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_p):
        super().__init__()
        kernel_size = 3

        self.model = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=1, padding=1),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.MaxPool2d(2, stride=2)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
model_cnn = nn.Sequential(
    ConvolutionalBlock(IMG_CHANNELS, 25, 0), # 25 x 14 x 14
    ConvolutionalBlock(25, 50, 0.2),  # 50 x 7 x 7
    ConvolutionalBlock(50, 75, 0),  # 75 x 3 x 3

    nn.Flatten(),
    nn.Linear(75 * 3 * 3, 512),
    nn.Dropout(.3),
    nn.ReLU(),

    nn.Linear(512, 24)
)

model_cnn = model_cnn.to(device)

In [ ]:
epochs = 20
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model_cnn.parameters())

cnn_res = train(model_cnn, train_loader, val_loader, loss_function, optimizer, epochs)


In [ ]:
cnn_test_loss, cnn_test_acc = test(model_cnn, test_loader, loss_function)

In [ ]:
plt.plot(cnn_res['train_loss'], label='Train Loss')
plt.plot(cnn_res['val_loss'], label='Validation Loss')
plt.axhline(cnn_test_loss, label='Test Loss', linestyle='--', color='r')
plt.title('CNN Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()